In [486]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [487]:
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [488]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

Using cuda device


In [489]:
# Define model
class ConvNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),          
            nn.Dropout(0.3),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.4),   
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.33), 
            # Ustaliłem, że dodatkowe warstwy w tym miejscu zaniżają dokładność i wydłużają uczenie
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model10 = ConvNet(num_classes=10).to(device)
print(model10)

ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.2, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.3, inplace=F

In [490]:
# Adadelta  - najgorsze ze wszystkich
# Adafactor - kiepskie
# Adagrad   - kiepskie
# Adam      - dobrze, ale Adam W. lepszy
# AdamW     - chyba najlepszy
# Adamax Implements Adamax algorithm (a variant of Adam based on infinity norm).
# NAdam - nieźle, ale ADAM ciutkę lepiej
# RAdam - nieźle, ale ADAM ciutkę lepiej
# Muon   -  kiepsko
# RMSprop - nieźle, ale < ADAM W.
# Rprop - kiepskie
# SGD stochastic gradient descent                           działa kiepsko
# ASGD Implements Averaged Stochastic Gradient Descent.     skoro SGD działa kiepsko to nie próbowałbym tego

# lista schedulerów:
# StepLR — redukuje lr o stałą wartość co x epok
# MultiStepLR — jak StepLR, ale redukcja w określonych z góry epokach (np. 10., 15., 20.)
# ExponentialLR — nie zagłębiam się w działaniu, płynniejszy niż poprzednie
# ConstantLR — raz zmniejsza/zwiększa, np. 0.01 0.01 0.01 0.005 0.005 0.005
# LinearLR — nieciekawe
# PolynomialLR - zmnienia wielomianowo
# CosineAnnealingLR — obniża lr wzdłuż krzywej cosinus do minimum, potem restart
# CosineAnnealingWarmRestarts — jak wyżej, ale z automatycznymi restartami co n epok gdyby nie osiągneło minimum
# CyclicLR
# OneCycleLR
# ReduceLROnPlateau — redukuje lr gdy loss (przy mode=min) nie spada lub accuracy nie rośnie
loss_fn = nn.CrossEntropyLoss()
start_lr=0.001
optimizer10 = torch.optim.AdamW(model10.parameters(), lr=start_lr)
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer10,
    mode='min',        
    factor=0.1,        
    patience=1,        
    min_lr=0.00001,
    threshold_mode='abs',    
    threshold=0.012,   
    cooldown=1        
)

In [491]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_value, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [492]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    accuracy = 100 * correct
    print(f"Test Error: \n Accuracy: {accuracy:>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return accuracy, test_loss

In [493]:
start_batch_size = 256
batch_schedule = {
    2:  128,
    5:  64,
}

def get_train_loader(batch_size):
    return DataLoader(training_data, batch_size=batch_size, shuffle=True)

def get_test_loader(batch_size):
    return DataLoader(test_data, batch_size=batch_size)

train_dataloader = get_train_loader(start_batch_size)
test_dataloader  = get_test_loader(start_batch_size)

In [494]:
epochs = 24
training_log = []

for t in range(epochs):
    print(f"Epoch {t+1} (CIFAR-10)\n-------------------------------")
    
    if(t in batch_schedule):
        current_batch_size = batch_schedule[t]
        train_dataloader = get_train_loader(current_batch_size)
        test_dataloader  = get_test_loader(current_batch_size)
        print("Batch size zmieniony na ",current_batch_size)

    train(train_dataloader, model10, loss_fn, optimizer10)
    accuracy, avg_loss = test(test_dataloader, model10, loss_fn)
    training_log.append({
        "epoch": t + 1,
        "accuracy": accuracy,
        "avg_loss": avg_loss,
    })
    lr_before = optimizer10.param_groups[0]['lr']
    scheduler.step(avg_loss)
    lr_after = optimizer10.param_groups[0]['lr']

    if lr_after != lr_before:
        print(f"Epoch {t+1}: lr zmienione {lr_before:.6f} → {lr_after:.6f}")
    else:
        print(f"Epoch {t+1}: lr = {lr_after:.6f}")
print("Done CIFAR-10")

Epoch 1 (CIFAR-10)
-------------------------------


loss: 2.409479  [  256/50000]
loss: 1.538226  [25856/50000]
Test Error: 
 Accuracy: 54.6%, Avg loss: 1.247947 

Epoch 1: lr = 0.001000
Epoch 2 (CIFAR-10)
-------------------------------
loss: 1.328611  [  256/50000]
loss: 1.159289  [25856/50000]
Test Error: 
 Accuracy: 63.7%, Avg loss: 1.009163 

Epoch 2: lr = 0.001000
Epoch 3 (CIFAR-10)
-------------------------------
Batch size zmieniony na  128
loss: 1.011771  [  128/50000]
loss: 1.071529  [12928/50000]
loss: 0.977100  [25728/50000]
loss: 1.034669  [38528/50000]
Test Error: 
 Accuracy: 70.7%, Avg loss: 0.834278 

Epoch 3: lr = 0.001000
Epoch 4 (CIFAR-10)
-------------------------------
loss: 0.769199  [  128/50000]
loss: 0.786251  [12928/50000]
loss: 0.811253  [25728/50000]
loss: 1.009594  [38528/50000]
Test Error: 
 Accuracy: 73.0%, Avg loss: 0.762905 

Epoch 4: lr = 0.001000
Epoch 5 (CIFAR-10)
-------------------------------
loss: 0.717665  [  128/50000]
loss: 0.915821  [12928/50000]
loss: 0.734719  [25728/50000]
loss: 0.693760  [

In [495]:
import json
import os

models_dir = "models10"
meta_dir="metadata10"

# sprawdzamy ile jest plików w folderze
n = len([name for name in os.listdir(models_dir) if os.path.isfile(os.path.join(models_dir, name))])

# jeśli jest n modeli, to nowy będzie cifar10_(n+1)
model_path = os.path.join(models_dir, f"cifar10_{n+1}.pth")
torch.save(model10.state_dict(), model_path)

training_metadata = {
    "network_architecture": str(model10),
    "optimizer": optimizer10.__class__.__name__,
    "learning_rate": start_lr,
    "scheduler": scheduler.__class__.__name__,
    "scheduler_params": {
        "mode": scheduler.mode,
        "factor": scheduler.factor,
        "patience": scheduler.patience,
        "min_lr": scheduler.min_lrs[0] if hasattr(scheduler, "min_lrs") else None,
        "threshold": scheduler.threshold,
        "cooldown": scheduler.cooldown,
    },
    "batch_size": start_batch_size,
    "training_log": training_log,
}

metadata_path = os.path.join(meta_dir, f"{n+1}.txt")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"Found {n} files in {models_dir}.")
print(f"Saved model to {model_path}")
print(f"Saved training metadata to {metadata_path}")

Found 38 files in models10.
Saved model to models10\cifar10_39.pth
Saved training metadata to metadata10\39.txt
